# 14 — TNG100 satellite anisotropy across host-mass cuts ($M_{*,\rm sat}>10^8\,M_\odot$)

Same machinery as `06_tng_anisotropy_z0_z0p05_logM8`, but instead of comparing simulations it
compares the different **host halo mass selections** built in notebooks 08–13, **TNG100 only**, at
**z = 0** and **z = 0.05**, for the fixed satellite cut $M_{*,\rm sat}>10^8\,M_\odot$.

Host-mass selections compared (all use the within-1 $R_{200c}$, $\log M_*>8$ catalogs):

| label | host cut [$\log_{10}M_{200c}$] | catalog file | built by |
|---|---|---|---|
| 12.0–12.5 | MW-mass (baseline) | `tng100_satellites_logM8.00.csv` | 01 |
| > 12      | open, $>12$        | `tng100_satellites_hostlogM12.0plus_logM8.00.csv` | 12 / 13 |
| 13–14     | $13<\log M<14$     | `tng100_satellites_hostlogM13.0-14.0_logM8.00.csv` | 08 / 09 |
| > 13      | open, $>13$        | `tng100_satellites_hostlogM13.0plus_logM8.00.csv` | 10 / 11 |

For each (host cut × redshift) it computes the anisotropy $P(\theta)$, the bootstrap quench
fraction $f_q(\theta)$, and the $f_q(\theta)=a+b\cos 2\theta$ MCMC fit, then overlays them.

> Run notebooks 08–13 first to produce the host-tagged catalogs. Any catalog that is missing on
> disk is skipped with a warning, so this notebook still runs on whatever subset is present.

In [ ]:
import os
import numpy as np
import pandas as pd
import emcee
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

# --- configuration -----------------------------------------------------------
LOGCUT    = 8                      # satellite log10(M*/Msun) cut: M*_sat > 1e8
SIM       = "tng100"              # TNG100 only
DATA_ROOT = "../data"

# host-mass selections: (label, filename tag prefix, color). The MW-mass baseline has no tag.
HOST_CUTS = [
    ("12.0-12.5", "",                    "#1f77b4"),   # MW-mass (notebook 01)
    (">12",       "hostlogM12.0plus_",   "#2ca02c"),   # notebooks 12 / 13
    ("13-14",     "hostlogM13.0-14.0_",  "#ff7f0e"),   # notebooks 08 / 09
    (">13",       "hostlogM13.0plus_",   "#d62728"),   # notebooks 10 / 11
]
# redshifts: (label, subdir, linestyle, marker)
REDS = [
    ("z=0",    "z0",    "-", "o"),
    ("z=0.05", "z0p05", "--", "s"),
]

# build the full (host cut x redshift) dataset list, TNG100 only
# each entry: (label, color, linestyle, marker, full_path)
DATASETS = []
for hlabel, tag, color in HOST_CUTS:
    for zlabel, zsub, ls, marker in REDS:
        fname = f"tng100_satellites_{tag}logM{LOGCUT:.2f}.csv"
        path  = os.path.join(DATA_ROOT, SIM, zsub, fname)
        DATASETS.append((f"{hlabel}, {zlabel}", color, ls, marker, path, hlabel, zlabel))

# 18 angle bins over [0, 90]
N_BINS        = 18
ANGLE_EDGES   = np.linspace(0, 90, N_BINS + 1)
ANGLE_CENTERS = 0.5 * (ANGLE_EDGES[:-1] + ANGLE_EDGES[1:])   # 2.5, 7.5, ..., 87.5

## Load the catalogs

Each catalog already carries its host selection and the `alpha` (0–90 deg) / `quenched` (1/0)
columns. We re-apply `mstar_phys > 8` explicitly so the satellite cut is self-documenting. Missing
files are skipped with a warning.

In [ ]:
data = {}   # label -> DataFrame  (only for catalogs that exist)
print(f"{'dataset':<18s} {'N_sat':>7s}  {'f_q':>6s}   file")
for label, color, ls, marker, path, hlabel, zlabel in DATASETS:
    if not os.path.exists(path):
        print(f"{label:<18s} {'--':>7s}  {'--':>6s}   MISSING: {path}  (run the matching 08-13 notebook)")
        continue
    df = pd.read_csv(path)
    df = df[df["mstar_phys"] > LOGCUT].reset_index(drop=True)   # M*_sat > 1e8
    data[label] = df
    print(f"{label:<18s} {len(df):7d}  {df['quenched'].mean():.3f}   {path}")

if not data:
    raise FileNotFoundError("no catalogs found -- run notebooks 08-13 (and 01) to build them first.")

## Helpers — anisotropy histogram, bootstrap $f_q$, MCMC sinusoid fit

Identical model to notebooks 03/06: $f_q(\theta)=a+b\cos 2\theta$ with a Gaussian jitter term
$\exp(f)$ added in quadrature to the per-bin error. $b=0$ is isotropic; $b>0$ means quenching is
enhanced toward the host major axis.

In [ ]:
def norm_hist(angles):
    """Probability density per degree over [0, 90] with N_BINS bins."""
    c, _ = np.histogram(angles, bins=ANGLE_EDGES)
    return c / c.sum() / (90.0 / N_BINS)

def bootstrap_fq(angle, quenched, N=10000, seed=0):
    """Binned-mean quench fraction per angle bin, with bootstrap mean and 1-sigma error."""
    rng = np.random.default_rng(seed)
    angle = np.asarray(angle); quenched = np.asarray(quenched, dtype=float)
    n = len(angle)
    boot = np.full((N, N_BINS), np.nan)
    bin_idx = np.digitize(angle, ANGLE_EDGES) - 1
    for i in range(N):
        s = rng.integers(0, n, n)
        bi, qi = bin_idx[s], quenched[s]
        for j in range(N_BINS):
            m = bi == j
            if m.any():
                boot[i, j] = qi[m].mean()
    return np.nanmean(boot, axis=0), np.nanstd(boot, axis=0)

def log_likelihood(theta, x, y, sigma):
    a, b, f = theta
    s = sigma ** 2 + np.exp(f) ** 2
    model = a + b * np.cos(2 * np.radians(x))
    return -0.5 * np.sum((y - model) ** 2 / s + np.log(2 * np.pi * s))

def log_prior(theta):
    a, b, f = theta
    return 0.0 if (0 < a < 1 and -1 < b < 1 and -10 < f < 2) else -np.inf

def log_prob(theta, x, y, sigma):
    lp = log_prior(theta)
    return lp + log_likelihood(theta, x, y, sigma) if np.isfinite(lp) else -np.inf

def fit_sinusoid(mean, std, n_walkers=20, n_steps=10000, burn=1000, seed=0):
    """MCMC fit of a + b*cos(2 theta); returns (params_mean, params_std)."""
    np.random.seed(seed)
    ok = np.isfinite(mean) & np.isfinite(std) & (std > 0)
    p0 = np.array([0.7, 0.025, -3.0]) + 1e-2 * np.random.randn(n_walkers, 3)
    sampler = emcee.EnsembleSampler(n_walkers, 3, log_prob,
                                    args=(ANGLE_CENTERS[ok], mean[ok], std[ok]))
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=burn, flat=True)
    return chain.mean(axis=0), chain.std(axis=0)

def model_band(p, e, n_mc=10000, seed=0):
    """Posterior median + 16/84th-percentile band of the sinusoid over [0, 90] deg."""
    rng = np.random.default_rng(seed)
    x = np.linspace(0, np.pi / 2, 400)
    a = rng.normal(p[0], e[0], n_mc); b = rng.normal(p[1], e[1], n_mc)
    y = a[:, None] + b[:, None] * np.cos(2 * x)[None, :]
    xdeg = np.degrees(x)
    return xdeg, (a.mean() + b.mean() * np.cos(2 * x)), np.percentile(y, 16, 0), np.percentile(y, 84, 0)

## Compute anisotropy, bootstrap, and MCMC fit for every dataset

`hist` holds $P(\theta)$; `fq` holds the bootstrap $(\text{mean},\sigma)$ per bin; `params` holds
the fitted $(a,b,f)$ with uncertainties. $|b|/\sigma_b$ is the amplitude significance.

In [ ]:
hist, fq, params = {}, {}, {}
for label in data:
    df = data[label]
    hist[label]   = norm_hist(df["alpha"].to_numpy())
    fq[label]     = bootstrap_fq(df["alpha"].to_numpy(), df["quenched"].to_numpy())
    params[label] = fit_sinusoid(*fq[label])

print(f"{'dataset':<18s}  {'a':>16s}   {'b':>16s}   |b|/sig_b")
for label, (p, e) in params.items():
    print(f"{label:<18s}  {p[0]:.3f} +/- {e[0]:.3f}   {p[1]:+.3f} +/- {e[1]:.3f}   "
          f"{abs(p[1] / e[1]):.2f}")

## A reusable two-panel plotter

Left = **anisotropy** $P(\theta)$. Right = **quench fraction vs angle** with the MCMC sinusoid fit
(points + error bars = bootstrap $f_q$; line + band = posterior median and 16–84th percentile).

In [ ]:
_style = {d[0]: d for d in DATASETS}   # label -> full tuple

def plot_panels(labels, title=None, legend_title=None):
    """Two-panel figure (anisotropy | f_q + sinusoid fit) overlaying the given dataset labels."""
    labels = [l for l in labels if l in data]   # skip any missing catalogs
    if not labels:
        print("(nothing to plot -- those catalogs are missing)"); return
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))
    for label in labels:
        _, color, ls, marker, *_ = _style[label]
        h = hist[label]
        axL.step(ANGLE_EDGES, np.r_[h, h[-1]], where="post", color=color, ls=ls, lw=2, label=label)
        m, s = fq[label]
        axR.errorbar(ANGLE_CENTERS, m, yerr=s, fmt=marker, color=color, capsize=3, ms=5,
                     ls="none", label=label)
        xdeg, ymed, ylo, yhi = model_band(*params[label])
        axR.plot(xdeg, ymed, color=color, ls=ls, lw=2)
        axR.fill_between(xdeg, ylo, yhi, color=color, alpha=0.12)
    for ax in (axL, axR):
        ax.set_xlim(0, 90); ax.set_xlabel(r"$\theta$ [deg]")
        ax.tick_params(which="both", direction="in", top=True, right=True)
    axL.set_ylim(0, 0.030); axL.set_ylabel(r"$P(\theta)$")
    axR.set_ylim(0, 1);     axR.set_ylabel(r"$f_q$")
    axL.set_title("Anisotropy"); axR.set_title("Quench fraction + MCMC fit")
    axL.legend(title=legend_title, fancybox=False, edgecolor="k")
    if title:
        fig.suptitle(title, y=1.02, fontsize=14)
    plt.subplots_adjust(wspace=0.22)
    plt.show()

## 1. Each dataset separately

In [ ]:
for label in data:
    plot_panels([label], title=label)

## 2. Each host-mass cut, both redshifts overlaid

In [ ]:
for hlabel, tag, color in HOST_CUTS:
    plot_panels([f"{hlabel}, z=0", f"{hlabel}, z=0.05"],
                title=f"TNG100, host {hlabel}", legend_title=f"host {hlabel}:")

## 3. All host-mass cuts at a fixed redshift

In [ ]:
for zlabel in ("z=0", "z=0.05"):
    plot_panels([f"{hlabel}, {zlabel}" for hlabel, *_ in HOST_CUTS],
                title=fr"TNG100, {zlabel}  ($M_{{*,\rm sat}}>10^8\,M_\odot$)",
                legend_title="host cut:")

## 4. Everything together

In [ ]:
plot_panels(list(data),
            title=r"TNG100, all host cuts, z = 0 and 0.05  ($M_{*,\rm sat} > 10^8\,M_\odot$)",
            legend_title="dataset:")

## 5. Anisotropy amplitude $b$ vs host-mass cut

Pulls the whole comparison into one figure: the fitted sinusoid amplitude $b$ (with its MCMC
$1\sigma$ error) for each host-mass cut, z=0 vs z=0.05. $b>0$ = quenching enhanced toward the major
axis; the dashed line marks isotropy ($b=0$).

In [ ]:
present = [h for h, *_ in HOST_CUTS if f"{h}, z=0" in data or f"{h}, z=0.05" in data]
x = np.arange(len(present))

fig, ax = plt.subplots(figsize=(8, 5))
for zlabel, dx, mk in [("z=0", -0.08, "o"), ("z=0.05", 0.08, "s")]:
    bs, es, xs = [], [], []
    for i, hlabel in enumerate(present):
        lab = f"{hlabel}, {zlabel}"
        if lab in params:
            p, e = params[lab]
            bs.append(p[1]); es.append(e[1]); xs.append(i + dx)
    ax.errorbar(xs, bs, yerr=es, fmt=mk, capsize=4, ms=7, lw=2, label=zlabel)
ax.axhline(0.0, color="grey", ls="--", lw=1)
ax.set_xticks(x); ax.set_xticklabels(present)
ax.set_xlabel(r"host-mass cut  [$\log_{10} M_{200c}$]")
ax.set_ylabel(r"sinusoid amplitude  $b$")
ax.set_title(r"Quenching anisotropy amplitude vs host mass ($M_{*,\rm sat}>10^8\,M_\odot$)")
ax.legend(fancybox=False, edgecolor="k")
ax.tick_params(which="both", direction="in", top=True, right=True)
plt.show()

# companion table
print(f"{'host cut':<12s} {'redshift':<8s} {'b':>10s} {'sig_b':>8s} {'|b|/sig_b':>10s}")
for hlabel in present:
    for zlabel in ("z=0", "z=0.05"):
        lab = f"{hlabel}, {zlabel}"
        if lab in params:
            p, e = params[lab]
            print(f"{hlabel:<12s} {zlabel:<8s} {p[1]:>+10.4f} {e[1]:>8.4f} {abs(p[1]/e[1]):>10.2f}")